# AETHER — Stage 5: Hidden-State Voice Head Training Probe

Обучаем `HiddenStateVoiceHead` (крошечный, 2 слоя, d_model=128) предсказывать Mimi codebook-0
токены **из hidden state Qwen3**, а не из текста — используя `kyutai/tts-1.6b-en_fr` как учителя
(подтверждено разведкой, см. `docs/colab.md`). ~20 английских фраз на обучение, 4 отложенные как
мягкое наблюдение (не статистически значимая проверка обобщения).

**Критерий успеха** — train loss падает от случайного старта (`checks.train_loss_decreased`), не
качество звучания. См. scope_note в самом report.json.

**GPU:** тяжелее предыдущих этапов — грузим одновременно Qwen3-1.7B и TTS-модель на 1.6B
параметров. T4 (16 ГБ) должен справиться (оба в bf16, ~3.4 ГБ каждая + Mimi), но если увидишь OOM,
переключись на GPU с бóльшим объёмом памяти (A100).

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
MODEL_ID = "Qwen/Qwen3-1.7B"  # @param {type:"string"}
TTS_HF_REPO = "kyutai/tts-1.6b-en_fr"  # @param {type:"string"}
VOICE_REPO = "kyutai/tts-voices"  # @param {type:"string"}
EPOCHS = "300"  # @param {type:"string"}

if "YOUR_USERNAME" in REPO_URL:
    raise ValueError("Укажи настоящий REPO_URL")


In [ ]:
import os, subprocess, sys
from pathlib import Path

subprocess.run(["nvidia-smi"], check=False)
repo_dir = Path("/content/aether")
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{repo_dir}[dev,ml,audio]"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "torch", "torchvision", "torchaudio"],
    check=True,
)
print("Commit:")
subprocess.run(["git", "rev-parse", "HEAD"], check=True)


In [ ]:
artifacts = repo_dir / "artifacts" / "colab-stage5"
artifacts.mkdir(parents=True, exist_ok=True)
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
(artifacts / "tests.log").write_text(tests.stdout, encoding="utf-8")
print(tests.stdout)
if tests.returncode != 0:
    raise RuntimeError("Tests failed")


In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(repo_dir / "src")
command = [
    sys.executable, "-m", "aether.experiments.colab_stage5",
    "--allow-download",
    "--model", MODEL_ID,
    "--tts-hf-repo", TTS_HF_REPO,
    "--voice-repo", VOICE_REPO,
    "--epochs", EPOCHS,
    "--output-dir", str(artifacts),
]
run = subprocess.run(command, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
(artifacts / "model_run.log").write_text(run.stdout, encoding="utf-8")
print(run.stdout)
print("Exit code:", run.returncode)


In [ ]:
import json

report_path = artifacts / "report.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print("Status:", report.get("status"))
print("Scope note:", report.get("scope_note"))
print("Checks:", report.get("checks"))
loss_curve = report.get("loss_curve", [])
if loss_curve:
    print("Loss: first =", loss_curve[0], " last =", loss_curve[-1], " min =", min(loss_curve))
print("Held-out token agreement:", report.get("held_out_token_agreement"))
print("Teacher diagnostics:", report.get("teacher_diagnostics"))


## Послушать train-фразы

Два файла на фразу:
- `teacher_voice_full` — настоящий голос учителя (Kyutai), все 32 кодбука. Должен звучать как
  обычная речь -- это доказывает, что вся инфраструктура (Mimi, TTSModel) реально работает.
- `hybrid_our_codebook0` -- та же реконструкция, но codebook 0 (единственное, что предсказывает
  наша `HiddenStateVoiceHead`) заменён на предсказание обученной головы, остальные 31 кодбук --
  от учителя. Если фраза всё ещё узнаваема после подмены -- голова выучила что-то осмысленное
  про содержание из hidden state. Если превращается в кашу -- не выучила. Это и есть честный
  тест "работает наша логика или нет", без побочных факторов декодирования через один codebook.


In [ ]:
from IPython.display import Audio, display

train_wav_files = report.get("train_wav_files", {})
for phrase_id, paths in list(train_wav_files.items())[:5]:
    print("===", phrase_id, "===")
    print("teacher voice (full 32 codebooks, real Kyutai speech):")
    display(Audio(filename=paths["teacher_voice_full"]))
    print("hybrid (teacher's codebooks 1-31 + OUR predicted codebook 0):")
    display(Audio(filename=paths["hybrid_our_codebook0"]))


## Если упало

`report.json` пишется на каждом шаге, даже при ошибке — `traceback.txt` в архиве покажет, где
именно (загрузка Qwen, загрузка TTS-модели, разрешение голоса, генерация учителя или обучение).
Пришли оба файла для разбора.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/aether-colab-stage5-logs", "zip", root_dir=artifacts)
print(archive)
files.download(archive)
